# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

Part 4 trained the foundation model. The recurring production job is to run it over every card and store a fresh embedding for downstream models to consume. That's a heterogeneous job — CPU to read the tokens, GPU for the forward pass — and it streams through a single Ray Data pipeline.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir, stale_or_missing
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Turn the trained model into one vector per transaction

The foundation model pretrained on long transaction histories, but the fraud model wants a fixed vector per transaction. Following NVIDIA's blueprint (their NB04) we embed each transaction **on its own** — feed `<bos>` + its field tokens + `<eos>` through the decoder and read the **last real token's** hidden state (`HuggingFaceDecoderInference`, `last_token` pooling).

Two choices happen here, and both matter:
- we **balance-sample the training set** (every fraud + a ~10% mix of normals) rather than embedding all ~19.5M train transactions, and embed the **100K stratified val/test eval subsets** from Part 2;
- the single-transaction context is what makes the embedding *complementary* to the raw features downstream (a 300-deep running average would just re-encode what the raw fields already say).

Each split is written as `embed_<split>.npy` + `lbl_<split>.npy` + `raw_<split>.parquet` (the 13 native features), so Part 6 can fit its classifiers without touching the tokenizer.

## Embed on a GPU worker with Ray

Embedding is GPU work, and the model is expensive to load — so we run it as a single Ray task pinned to a GPU (`@ray.remote(num_gpus=1)`), loading the decoder **once** and streaming transactions through it in large batches (`extract_embeddings_batched`). Ray schedules it on a GPU worker off the head node and autoscales one up if needed; `src/nvembed.py` wraps this so the notebook and a headless job embed identically. At `full`, embedding the 100K eval subsets + the 1M balanced train sample is a few minutes on one A10G; larger corpora would fan the splits across more GPU workers.

In [ ]:
from src.nvembed import embed_splits

eb = cfg["embed"]
# Embed each transaction ON ITS OWN with our pretrained decoder (Part 4), via NVIDIA's
# HuggingFaceDecoderInference: <bos> + the transaction's field tokens + <eos>, then read the
# last real token's hidden state. We balance-sample the training set (~10% fraud) and embed
# the 100K stratified val/test eval subsets — writing embed_/lbl_/raw_ per split for Part 6.
if not os.path.exists(os.path.join(paths["embeddings"], "embed_test.npy")):
    meta = embed_splits(
        hf_dir=paths["hf"],                          # our exported decoder (Part 4)
        split_dir=paths["nvsplit"],                  # native TabFormer splits (Part 2)
        out_dir=paths["embeddings"],
        balanced_train=eb["balanced_train"],
        max_length=eb.get("embed_max_length", 128),  # single-transaction context (NVIDIA MAX_LENGTH)
        batch_size=eb["batch_size"],
    )
    print(json.dumps(meta, indent=2))
else:
    print(f"embeddings present at {paths['embeddings']} — skipping (delete the dir to re-embed)")

## Did the embeddings actually learn anything?

The classic self-supervised failure is silent **representation collapse**: every customer maps to nearly the same vector while the loss looks fine. A cheap guard is to sample the embeddings and check two numbers — mean pairwise cosine similarity (→1.0 means collapse) and mean feature variance (→0 means collapse). `embedding_health` reports both.

In [ ]:
# Representation-collapse check on the test embeddings: mean pairwise cosine similarity
# (→1.0 = collapse) and mean feature variance (→0 = collapse).
emb = np.load(os.path.join(paths["embeddings"], "embed_test.npy"))
lbl = np.load(os.path.join(paths["embeddings"], "lbl_test.npy"))

rng = np.random.RandomState(0)
idx = rng.choice(len(emb), size=min(2000, len(emb)), replace=False)
s = emb[idx].astype(np.float32)
s /= (np.linalg.norm(s, axis=1, keepdims=True) + 1e-8)
cos = s @ s.T
mean_cos = float((cos.sum() - len(idx)) / (len(idx) * (len(idx) - 1)))

print(f"{len(emb):,} test embeddings · dim {emb.shape[1]}  ({int(lbl.sum()):,} fraud)")
print(f"mean pairwise cosine  {mean_cos:.3f}   (→1.0 = collapse)")
print(f"mean feature variance {float(emb.var(0).mean()):.4f}   (→0 = collapse)")
print(f"example embedding[:8] = {emb[0, :8].round(3).tolist()}  (label {int(lbl[0])})")

## Takeaways

**Ray**
- Embedding runs as one GPU task (`@ray.remote(num_gpus=1)` in `src/nvembed.py`) that loads the decoder once and batches through it — scheduled off the head node, autoscaled on demand. The notebook and `scripts/nvidia_repro` embed identically.

**The embeddings**
- One vector per transaction, from the **last real token** of a single-transaction sequence — aligned with the per-transaction fraud label and complementary to the raw fields.
- Written per split as `embed_/lbl_/raw_` so Part 6 fits its classifiers with no re-tokenization.
- A quick collapse check (mean pairwise cosine, feature variance) catches the silent self-supervised failure mode before it reaches the downstream model.

---

## Next

**Part 6 — Downstream fraud: raw vs FM vs fusion**: fit NVIDIA's XGBoost recipe on three feature sets — raw transaction fields, the FM embedding, and the two fused — and measure the lift at natural fraud prevalence with PR-AUC.